# 04 · The four extinction tests
### Machine Unlearning is Extinction, Not Forgetting

Behavioural neuroscience shows that *extinction* hides a memory rather than erasing it, because the memory
comes back in four characteristic ways. This notebook tests each of them on every unlearned model, and on
the retrained model as the "truly forgotten" baseline.

| Phenomenon | Animal version | What we do (never showing forget data, except in savings) | Evidence of extinction |
|---|---|---|---|
| **Spontaneous recovery** | response returns with time | (a) keep training on retain data; (b) weight noise; (c) quantization; (d) pruning | forget accuracy rises |
| **Renewal** | response returns in another context | evaluate in new contexts (gray, blur, noise, contrast, CIFAR-100-C); ABA models: gray (B) vs colour (A) | forget accuracy higher outside the unlearning context |
| **Reinstatement** | a related cue brings it back | fine-tune briefly on **sibling** classes vs **similar** vs **dissimilar** classes | recovery is larger for related cues |
| **Rapid reacquisition** | relearning is faster than first learning | relearn from 5 / 20 / 100 forget images | faster than the retrained model |

### BatchNorm controls (why every fine-tuning test runs twice)
"The BatchNorm illusion" (Kalani et al., 2026) showed that just recomputing a model's BatchNorm statistics,
with no weight change, can reverse apparent forgetting. Every fine-tuning test would update those statistics
too, so a reviewer could attribute any recovery to this artifact. We therefore:
1. run every fine-tuning test with BatchNorm statistics **updating** and **frozen** (column `bn`);
2. add a **statistics-only** proxy `bn_recal` (no weight change) as a pure measure of the artifact;
3. offer an optional **GroupNorm arm** (no running statistics at all; see notebook 01).

The primary tests in notebook 06 use frozen statistics (plus the weight-perturbation proxies, marked
`bn = eval`), so any recovery they find lives in the weights.

**Multi-context models** (from notebook 03, stage E) get all four tests too, for the remedy test H6.

**Specificity control.** Every accuracy is also reported as `chance_corrected` = forget-class recall minus
the rate at which *other* images are wrongly called the forget class. Real recovery of knowledge raises the
first number. Mere drift towards predicting that label raises both, and cancels out.

### Two stages
- **P, the pilot** (a few minutes): all tests on the pilot-class models. Decide **go / no-go** and choose the
  primary spontaneous-recovery proxy *before* looking at the main data.
- **M, the main run** (~8–10 h on T4 x2, so expect **two sessions**: re-run with this output attached): all tests on all models.

## Kaggle settings for this notebook

Set these in the right-hand **Settings** panel (in the editor: *View → Show sidebar* if it is hidden).

| Setting | Value | Why |
|---|---|---|
| **Accelerator** | **GPU T4 x2** | Two GPUs: `ext_utils` runs one independent worker per GPU, so two models train at the same time. Your weekly quota is counted in session hours, not per GPU, so T4 x2 roughly doubles what you get per hour. (A single P100 also works: the code detects one GPU and uses one worker.) |
| **Internet** | **On** | Needed to download CIFAR-100 (~170 MB) and, in notebook 04, CIFAR-100-C, unless you attach them as datasets. Internet requires a phone-verified Kaggle account. |
| **Persistence** | Files only (optional) | Keeps `/kaggle/working` between *interactive* sessions, so you don't lose files if the browser disconnects. Not needed for background runs. |
| **Environment** | *Always use latest* for the first run, then **Pin to original environment** | Pinning freezes the library versions (PyTorch etc.), so a Kaggle image update can't break a half-finished experiment. |
| **Language** | Python | |

**Inputs to attach** (*Add Input* → *Your Work* / *Your Datasets*):

| Input | Why |
|---|---|
| `ext-code` dataset (contains `ext_utils.py`) | The shared library. Upload it once as a private dataset. |
| **CIFAR-100 Python** dataset ([fedesoriano/cifar100](https://www.kaggle.com/datasets/fedesoriano/cifar100)) | **Strongly recommended.** Without it CIFAR-100 is downloaded from cs.toronto.edu at ~50 kB/s (56 min in run #1). Check that the setup cell prints `CIFAR-100: found at …`. |
| Outputs of **notebooks 01, 02, 03** (`ext-01-originals`, `ext-02-retrain`, `ext-03-unlearning`) | The models to test (original, retrained, unlearned, ABA) and their metrics. |
| **CIFAR-100-C** dataset ([rojanregmi1/cifar100-c](https://www.kaggle.com/datasets/rojanregmi1/cifar100-c)) | **Recommended.** Otherwise a ~2.9 GB archive is streamed from Zenodo, which may be as slow as the CIFAR-100 server was in run #1. The setup prints `[CIFAR-100-C] using …` when it is found (it must contain `labels.npy` and e.g. `fog.npy`). |
| *(when resuming)* this notebook's previous output (same name as below) | Skips finished jobs. |

**How to run.** Click **Save Version → Save & Run All (Commit)**. The notebook then runs in the background
for up to 12 hours with the browser closed. When it finishes, its files appear under *Output*, and the next
notebook attaches them as an input. 

**If a run stops early** (time budget or error), nothing is lost. Every finished job is written to disk
immediately. Attach this notebook's own latest output as an input and run it again: finished jobs are
skipped and their files are carried over.

## Setup: import the shared library

This cell finds `ext_utils.py` and imports it as `U`. All the real code (data pipeline, model, unlearning
methods, tests, job runner) lives there, so every notebook uses *identical* code.

`U.env_report()` then prints:
- the Python and PyTorch versions,
- which GPUs are visible (you should see **two Tesla T4** with the T4 x2 accelerator),
- free disk space,
- whether the internet is reachable,
- which inputs are attached.

**Check this output before going further.** If it says `GPU: none found`, the accelerator is off. If it
says `internet: OFF` and no CIFAR-100 dataset is attached, data loading will fail.

In [ ]:
# Locate ext_utils.py (from an attached dataset, a utility script, or the working folder) and import it.
import os, sys, glob
candidates = (glob.glob("/kaggle/working/ext_utils.py")
              + glob.glob("/kaggle/usr/lib/**/ext_utils.py", recursive=True)
              + glob.glob("/kaggle/input/**/ext_utils.py", recursive=True)
              + glob.glob(os.path.join(os.getcwd(), "ext_utils.py"))
              + glob.glob(os.path.join(os.getcwd(), "..", "ext_utils.py")))
if not candidates:
    raise FileNotFoundError(
        "ext_utils.py not found. Upload it as a private Kaggle Dataset (e.g. 'ext-code') and attach it "
        "with 'Add Input' -> 'Your Datasets' (see the Kaggle settings section above).")
sys.path.insert(0, os.path.dirname(os.path.abspath(candidates[0])))
import ext_utils as U
print("ext_utils loaded from:", U.UTILS_FILE)
U.env_report()

## Configuration

| Parameter | Value | What it does and why this value |
|---|---|---|
| `smoke` | `False` | `True` = a tiny dry run (30 training and 10 test images per class, 1 epoch, width 16) that finishes in minutes. Its files get a `_smoke` suffix, so they never mix with real results. Use it once to test the whole chain, then set `False`. **Must be the same in every notebook.** |
| `seeds` | `[0, 1, 2]` | Independent repetitions (different weight initialisation and data order). Three is the minimum for estimating variance. One statistical unit = (method, class, seed). |
| `forget_classes` | 10 classes | One class from each of 10 *different* CIFAR-100 superclasses, so no single superclass drives the result. The other 4 classes in each superclass are its "siblings" (semantic neighbours). |
| `norm` | `"bn"` | Normalization layer. `"bn"` = BatchNorm, the standard ResNet-18 of the unlearning literature (main experiments). `"gn"` = GroupNorm, a **control arm without running statistics**. The "BatchNorm illusion" (Kalani et al. 2026) showed that recomputing BatchNorm statistics alone can undo apparent forgetting. If the extinction signs also appear with GroupNorm, they are not a BatchNorm artifact. GroupNorm files get a `_gn` suffix, so both arms can live side by side. **Must be the same in every notebook of one arm.** |
| `pilot_class` | `"lamp"` | Used only for hyper-parameter calibration and the go/no-go pilot, then **excluded** from the main analysis. That way hyper-parameters are never tuned on the data that is later tested. |
| `width`, `epochs`, … | as in notebook 01 | Must match, so that the checkpoints load correctly (checked automatically). |
| `methods` | 5 methods | The methods whose unlearned models are tested. |
| `tests` | all 4 | Which extinction tests to run on each unlearned, multi-context and retrained model. The original models and the ABA models only get the context test, which is all they are needed for. |
| `recovery_bn_modes` | `["update", "frozen"]` | **BatchNorm control 1.** Every test that fine-tunes (retain fine-tuning, reinstatement, savings) trains **once, normally**, and scores each checkpoint twice: `update` = with the BatchNorm statistics that training produced, `frozen` = with the **original** statistics put back, so only the weight change counts. Recovery that survives in `frozen` cannot be the BatchNorm illusion. Notebook 06 uses `frozen` for the primary tests. (Until v1.1 the models were trained with BatchNorm layers in eval mode; in the pilot, run #4, that made several models collapse to chance.) |
| `collapse_frac` | 0.5 | A fine-tuning row whose accuracy on the kept classes falls below half its starting value is marked `collapsed` and left out of every analysis: a diverged network says nothing about hidden knowledge. The pilot read-out shows how many rows were affected. |
| `bn_recal_sizes` | [10, 100, 1000, 10000] | **BatchNorm control 2.** Recompute the BatchNorm statistics from this many retain images, with **no weight change** at all. Kalani et al. (2026) found ~10 images enough to undo much apparent forgetting. This measures how much "recovery" normalization statistics alone produce. |
| `sr_epochs` | 5 | **Spontaneous recovery, proxy (a):** epochs of continued training on *retain data only*, a stand-in for "time passing" during benign further use of the model. The forget class is never shown. |
| `sr_lr` | 0.01 | Learning rate of that continued training. It is 10× below the training peak: benign fine-tuning, not re-training. |
| `sr_noise` | [0.005, 0.01, 0.02, 0.05] | **Proxy (b):** Gaussian noise added to every conv and linear weight, with std = value × that tensor's own std. Small perturbations test whether the suppression sits in a fragile spot. |
| `sr_noise_reps` | 3 | Noise draws per level, averaged, so a single lucky draw can't drive the result. |
| `sr_quant_bits` | [8, 6, 4] | **Proxy (c):** per-channel weight quantization. Quantization has been reported to undo LLM unlearning; this tests the same for vision models. |
| `sr_prune` | [0.2, 0.5, 0.7] | **Proxy (d):** share of the smallest weights set to zero (global magnitude pruning). |
| `ctx_synthetic` | gray, blur, noise, contrast | **Renewal (ABC):** new input contexts made on the GPU. They need no download and always run. |
| `c_corruptions` | 6 corruptions | **Renewal (ABC)** with the standard CIFAR-100-C corruptions: noise, blur, weather, digital and brightness families. |
| `c_severities` | [3, 5] | A medium and the strongest corruption level. |
| `download_cifar_c` | `True` | If CIFAR-100-C is not attached, stream the Zenodo archive (~2.9 GB) and keep only the 6 files we need (~0.9 GB, in `/tmp`, not saved to the output). If the download fails, only the synthetic contexts run and the notebook continues. |
| `ri_n_classes` | 4 | **Reinstatement:** number of cue classes per condition. 4 = all superclass siblings. |
| `ri_epochs`, `ri_lr`, `ri_batch_size` | 3, 0.01, 128 | Short, gentle fine-tuning on the cue classes plus an equal-sized replay set of other retain classes. The replay prevents the model from collapsing onto only the 4 cue classes. The forget class is never shown. |
| `sv_shots` | [5, 20, 100] | **Savings:** number of forget-class images available for relearning (few-shot to moderate). |
| `sv_steps`, `sv_lr`, `sv_replay_bs` | 100, 0.01, 128 | Relearning steps. Each step uses all the shots plus 128 replay images, so the rest of the model isn't destroyed. |
| `sv_eval_steps` | 0, 1, 2, 5, 10, 20, 50, 100 | When forget accuracy is measured along the relearning curve. Log-spaced, because relearning is fastest at the start. |
| `time_budget_h` | 11.0 | Kaggle stops sessions at 12 h. The runner stops *starting* new jobs after ~11 h (it estimates each job's length), so the notebook finishes cleanly and its output is saved. Re-run to continue. |
| `n_workers` | `None` | `None` = one worker process per visible GPU (2 on T4 x2, 1 on P100). Set 1 to force a single worker, e.g. to see tracebacks directly. |
| `reuse_ckpt` | `True` | If a job's checkpoint already exists (e.g. saved just before a crash), load it instead of recomputing it. |
| `raise_on_error` | `True` | After all jobs have run, the cell raises an error if any job failed, so failures can't go unnoticed. Finished jobs are always kept. |
| `poll_seconds` | 60 | How often the progress line (`done/total`, elapsed minutes, last log line of each worker) is printed. |
| `RUN_PILOT`, `RUN_MAIN` | `True`, `False` | Run the pilot first. Look at its table, record the go/no-go decision, then set `RUN_MAIN = True` and run the whole notebook as a new version. |

In [ ]:
CFG = U.make_cfg(
    smoke=False,
    seeds=[0, 1, 2],
    forget_classes=["maple_tree", "tiger", "rose", "shark", "bus",
                    "apple", "chair", "butterfly", "girl", "castle"],
    pilot_class="lamp",
    norm="bn",                      # "gn" = GroupNorm control arm (see notebook 01)
    width=64, epochs=40, batch_size=256, lr=0.1, pct_start=0.25,
    momentum=0.9, nesterov=True, weight_decay=5e-4, label_smoothing=0.0,
    amp=True, channels_last=True, cudnn_benchmark=True, eval_batch_size=1024,
    ckpt_fp16=True, retain_eval_subset=5000,
    methods=["FT", "NegGrad+", "RL", "SCRUB", "SalUn"],
    aba_classes=["maple_tree", "tiger", "rose", "shark", "bus"], aba_context="gray",
    mctx_classes=["maple_tree", "tiger", "rose", "shark", "bus"], mctx_contexts=["clean", "gray", "blur"],
    # extinction tests
    tests=["spontaneous", "contexts", "reinstatement", "savings"],
    recovery_bn_modes=["update", "frozen"], bn_recal_sizes=[10, 100, 1000, 10000],
    sr_epochs=5, sr_lr=0.01, sr_noise=[0.005, 0.01, 0.02, 0.05], sr_noise_reps=3,
    sr_quant_bits=[8, 6, 4], sr_prune=[0.2, 0.5, 0.7],
    ctx_synthetic=["gray", "blur", "noise", "contrast"],
    c_corruptions=["gaussian_noise", "motion_blur", "fog", "contrast", "jpeg_compression", "brightness"],
    c_severities=[3, 5], download_cifar_c=True,
    ri_epochs=3, ri_lr=0.01, ri_n_classes=4, ri_batch_size=128,
    sv_shots=[5, 20, 100], sv_steps=100, sv_lr=0.01, sv_replay_bs=128,
    sv_eval_steps=[0, 1, 2, 5, 10, 20, 50, 100], collapse_frac=0.5,
    # infrastructure
    time_budget_h=11.0, n_workers=None, reuse_ckpt=True, raise_on_error=True, poll_seconds=60,
)
RUN_PILOT, RUN_MAIN = True, False
SHARD = None                    # main run on two accounts: (0, 2) on one, (1, 2) on the other
U.check_experiment_config(CFG)

## CIFAR-100-C

This looks for an attached CIFAR-100-C dataset first and otherwise downloads the archive once, before the
workers start. If neither works, you'll see a message: the six CIFAR-100-C contexts are then skipped, and
everything else still runs.

In [ ]:
if "contexts" in CFG["tests"]:
    U.prefetch_cifar_c(CFG)

## Stage P: pilot on the `lamp` models

In [ ]:
if RUN_PILOT:
    U.launch("extinction", U.jobs_extinction(CFG, pilot=True), CFG)

### Pilot read-out and go/no-go

**How recovery is scored (v1.2).** A test "recovers" the forgotten class when its score goes up after the
manipulation. Every table is *after minus before*, per method; `retrain` is the truly-forgotten reference.

| Score | Meaning | Why |
|---|---|---|
| `top5_cc` (**primary**) | share of forget-class test images with the forgotten class among the model's **top 5** answers, minus the same share on other images | graded: partial recovery counts before the class wins outright |
| `chance_corrected` | the same with top-1 (the class must be the model's answer) | the strict version; it has a floor, which is why every pilot value in run #4 was exactly 0 |
| `rank_score` | where the forgotten class ranks among the 100 answers on its own images, rescaled to 0..1 (1 = first) | fully graded |
| `forget_auc` | how well the forgotten class's output separates its images from all others (0.5 = chance) | ignores a uniform suppression of that output |

**BatchNorm column.** `update` = statistics produced by the fine-tuning, `frozen` = original statistics put
back (weight change only), `recal` = statistics recomputed with no weight change (the pure BatchNorm
artifact), `eval` = weights perturbed with statistics fixed.

**Sanity first.** The first table shows the lowest accuracy on the kept classes that each fine-tuning test
reached. Rows below half the starting value are `collapsed` and excluded; there should be few or none.

**Go** if at least one method's primary score, or its savings AUC, beats the retrained model by more than
0.05 somewhere (the summary lists these). **No-go** if nothing moves: still a valid negative finding, but
then rethink the design before spending the main budget.

The cell then proposes the spontaneous-recovery proxy with the largest pilot recovery on the primary score
(among `frozen`/`eval` rows, never `bn_recal`) as the **primary H1 test**, and writes it with the primary
score to `analysis_choices.json`, fixing both *before* the main data exists. Notebook 06 reads them from there.


In [ ]:
import numpy as np, pandas as pd
df = U.load_df("extinction", CFG)
p = df[df["cls"] == CFG["pilot_class"]] if len(df) else df
if len(p) == 0:
    print("No pilot results yet.")
else:
    ids = ["kind", "method", "cls", "seed"]
    PM = U.PRIMARY_METRIC
    col = p["collapsed"].fillna(False).astype(bool) if "collapsed" in p else pd.Series(False, index=p.index)
    ft = p[p["test"].isin(["spontaneous", "reinstatement", "savings"]) & p["bn"].isin(["update", "frozen"])]
    print(f"SANITY: {int(col[ft.index].sum())} of {len(ft)} fine-tuning rows collapsed "
          f"(accuracy on kept classes < {CFG['collapse_frac']} x its starting value). Lowest accuracy reached:")
    display(ft.groupby(["test", "bn", "method"])["retain_test_acc"].min().unstack("method").round(3))
    ok = p[~col]
    base = ok[ok["test"] == "base"][ids + U.RESPONSE_METRICS]

    def rec(d, metric):
        d = d.merge(base[ids + [metric]].rename(columns={metric: "before"}), on=ids)
        return d.assign(recovery=d[metric] - d["before"])

    def last_epoch(d):
        return d[d["level"] == d["level"].max()]

    def excess(d, keys):
        """Largest (method minus retrain) mean recovery over the conditions in `keys`, per method."""
        t = d.groupby(["method"] + keys)["recovery"].mean().unstack("method")
        if "retrain" not in t:
            return pd.Series(dtype=float)
        return t[[m for m in CFG["methods"] if m in t]].sub(t["retrain"], axis=0).max()

    summary = {}
    for metric in U.RESPONSE_METRICS:
        sr = rec(ok[ok["test"] == "spontaneous"], metric)
        ctx = rec(ok[ok["test"] == "context"], metric)
        ri = rec(last_epoch(ok[ok["test"] == "reinstatement"]), metric)
        if metric == PM:
            print(f"\n=== PRIMARY score: {PM} ===")
            print("Spontaneous recovery (after - before):")
            display(sr.pivot_table(index=["proxy", "bn", "level"], columns="method", values="recovery").round(3))
            print("Renewal (context minus clean):")
            display(ctx.pivot_table(index=["context", "severity"], columns="method", values="recovery").round(3))
            print("Reinstatement (after the last epoch):")
            display(ri.pivot_table(index=["condition", "bn"], columns="method", values="recovery").round(3))
            recal = sr[(sr["proxy"] == "bn_recal") & (sr["kind"] == "unl")]
            if len(recal) and recal["recovery"].max() > 0.05:
                print(f"NOTE: BatchNorm recalibration alone restores up to {recal['recovery'].max():.3f} - "
                      "the BatchNorm illusion is present; rely on the 'frozen' rows.")
        w = sr[sr["bn"].isin(["frozen", "eval"]) & (sr["proxy"] != "bn_recal")]
        summary[(metric, "spontaneous")] = excess(w, ["proxy", "level"])
        summary[(metric, "renewal")] = excess(ctx, ["context", "severity"])
        summary[(metric, "reinstatement")] = excess(ri[ri["bn"] == "frozen"], ["condition"])

    sv = p[p["test"] == "savings"]
    if len(sv):
        bad = set(map(tuple, sv[col[sv.index]][["method", "shots", "bn"]].drop_duplicates().values))
        sv = sv[[(m, s, b) not in bad for m, s, b in zip(sv["method"], sv["shots"], sv["bn"])]]
        print("\nSavings (relearning from few forget images): top-1 forget accuracy, original BN statistics (frozen):")
        display(sv[sv["bn"] == "frozen"].pivot_table(index=["shots", "level"], columns="method",
                                                     values="forget_test_acc").round(3))
        trap = getattr(np, "trapezoid", None) or np.trapz

        def _auc(g):
            g = g.sort_values("level"); x = g["level"].astype(float).values; y = g["forget_test_acc"].values
            return float(trap(y, x) / (x.max() - x.min())) if len(x) > 1 and x.max() > x.min() else float(y.mean())

        a = pd.DataFrame([dict(method=k[0], shots=k[1], bn=k[2], auc=_auc(g))
                          for k, g in sv.groupby(["method", "shots", "bn"])])
        ref = a[a["method"] == "retrain"][["shots", "bn", "auc"]].rename(columns={"auc": "auc_ref"})
        a = a[a["method"] != "retrain"].merge(ref, on=["shots", "bn"]).assign(excess=lambda t: t["auc"] - t["auc_ref"])
        print("Savings: relearning AUC minus the retrained model's (positive = relearns faster):")
        display(a.pivot_table(index=["bn", "shots"], columns="method", values="excess").round(3))
        summary[("top-1 AUC", "savings")] = a[a["bn"] == "frozen"].groupby("method")["excess"].max()

    table = pd.DataFrame(summary).T
    table.index.names = ["score", "test"]
    print("\nSUMMARY - best advantage over the retrained model (> 0.05 = signal):")
    display(table.round(3))
    hits = [(i, m) for i, row in table.iterrows() for m, v in row.items() if pd.notna(v) and v > 0.05]
    prim_hits = [h for h in hits if h[0][0] in (PM, "top-1 AUC")]
    if prim_hits:
        print("GO signal on the primary score or savings:", ", ".join(f"{m} ({i[1]})" for i, m in prim_hits))
    else:
        print("NO signal on the primary score or savings. Discuss before starting the main run.")

    cand = rec(ok[(ok["test"] == "spontaneous") & (ok["kind"] == "unl") & ok["bn"].isin(["frozen", "eval"])
                  & (ok["proxy"] != "bn_recal")], PM)
    unl = cand.groupby(["proxy", "level"])["recovery"].mean()
    if len(unl) and unl.max() > 0:
        best = unl.idxmax()
        print(f"Suggested primary H1 proxy: {best[0]} at level {float(best[1])} (mean pilot recovery {unl.max():.3f})")
        U.save_choices(CFG, primary_metric=PM, primary_sr_proxy=best[0], primary_sr_level=float(best[1]))
    else:
        print("No proxy shows positive pilot recovery - keeping the default primary proxy "
              f"('retain_ft', {CFG['sr_epochs']}). Consider this in your go/no-go decision.")
        U.save_choices(CFG, primary_metric=PM, primary_sr_proxy="retain_ft", primary_sr_level=float(CFG["sr_epochs"]))


## Stage M: main run

Set `RUN_MAIN = True` (and optionally `RUN_PILOT = False`), then **Save & Run All**. Every job loads one model
and runs its tests. Unlearned, multi-context and retrained models get all four tests; original and ABA models
get the context test only.

Keep `RUN_PILOT = True` and attach **this notebook's pilot output**: its 7 jobs are then skipped in seconds and
`analysis_choices.json` is written into the new output again. If the run hits the time budget, attach that
output and run again; finished jobs are skipped.

**Splitting over two accounts:** set `SHARD = (0, 2)` in one copy and `SHARD = (1, 2)` in the other. Each
does half the models. Name the outputs `ext-04-extinction-a` and `-b` and attach both to notebook 06.

In [ ]:
if RUN_MAIN:
    jobs = U.jobs_extinction(CFG, pilot=False, shard=SHARD)
    print(f"{len(jobs)} models to test")
    U.launch("extinction", jobs, CFG)

## Quick look at the main results

Mean scores of the untouched models (before any test), per kind and method. This is only a preview; the full
statistics are in notebook 06.


In [ ]:
df = U.load_df("extinction", CFG)
main = df[(df["cls"] != CFG["pilot_class"])] if len(df) else df
if len(main):
    print("rows:", len(main), "| models:", main["key"].nunique())
    display(main[main["test"] == "base"].groupby(["kind", "method"])[["forget_test_acc", "top5_cc", "forget_rank", "forget_auc", "retain_test_acc"]].mean().round(4))
else:
    print("No main results yet (set RUN_MAIN = True).")

## Next step

Save a version. Open **notebook 05** (mechanism) with the outputs of 01, 02 and 03 attached, and later
**notebook 06** with the outputs of all notebooks attached.

## Name of this run's output

Title this notebook **`ext-04-extinction`**. When the run has finished, create a dataset from its output with the
same name: open the notebook's viewer page → **Output** → **New Dataset** → title **`ext-04-extinction`**, Private.
(Or skip the dataset and attach the notebook output directly: *Add Input → Your Work → Notebooks*.)

| Output name | Contains | Attached by |
|---|---|---|
| `ext-04-extinction` | `extinction` results (pilot + main), `analysis_choices.json` | notebook 06 |

- **Re-runs and resumes:** don't create a new name. Add a **New Version** of the same dataset. Each output
  already contains everything from earlier runs (it's carried over), so only the latest version matters.
  In the notebooks that use it, update the input to the new version.
- **Smoke (dry) runs:** add `-smoke`, e.g. `ext-04-extinction-smoke`, so real and test outputs never share a dataset.
- **GroupNorm control arm** (`norm="gn"`): add `-gn`, e.g. `ext-04-extinction-gn`.
- Results written by ext_utils 1.1 (the first pilot, run #4) are ignored automatically: they used the collapsing BatchNorm training and have no graded scores. Re-run the pilot with 1.2.
- The pilot run and the main run are two versions of the **same** dataset `ext-04-extinction` (the main run carries the pilot results over). The main run will likely need two sessions; each re-run is another version of the same dataset.